In [ ]:
"""
Random Forest Model for Predicting Flexural Strength of Gypsum Composites
Author: Haseeb Ahmad
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.inspection import permutation_importance


# ==========================
# 1. Load and Prepare Data
# ==========================

def load_data(file_path):
    df = pd.read_csv(file_path, header=2)
    df_array = np.asarray(df)

    # Input Features
    X = np.column_stack([
        df_array[0:161, 4].astype(float),
        df_array[0:161, 5].astype(float),
        df_array[0:161, 6].astype(float),
        df_array[0:161, 7].astype(float),
        df_array[0:161, 8].astype(float),
        df_array[0:161, 9].astype(float),
        df_array[0:161, 10].astype(float)
    ])

    # Target
    y = df_array[0:161, 12].astype(float)

    # Remove NaN values
    mask = ~np.isnan(y)
    X = X[mask]
    y = y[mask]

    return X, y


# ==========================
# 2. Train Model
# ==========================

def train_random_forest(X_train, y_train):
    model = RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    return model


# ==========================
# 3. Evaluate Model
# ==========================

def evaluate_model(model, X_train, X_test, y_train, y_test):

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    results = {
        "Train R2": r2_score(y_train, y_train_pred),
        "Test R2": r2_score(y_test, y_test_pred),
        "Train RMSE": np.sqrt(mean_squared_error(y_train, y_train_pred)),
        "Test RMSE": np.sqrt(mean_squared_error(y_test, y_test_pred)),
        "Train MAE": mean_absolute_error(y_train, y_train_pred),
        "Test MAE": mean_absolute_error(y_test, y_test_pred),
    }

    return results, y_test_pred


# ==========================
# 4. Plot Functions
# ==========================

def plot_convergence(X_train, y_train, X_test, y_test, save_path=None):

    n_estimators_range = [1, 5, 10, 20, 50, 100, 150, 200]
    train_scores = []
    test_scores = []

    for n in n_estimators_range:
        model = RandomForestRegressor(n_estimators=n, random_state=42, n_jobs=-1)
        model.fit(X_train, y_train)

        train_scores.append(r2_score(y_train, model.predict(X_train)))
        test_scores.append(r2_score(y_test, model.predict(X_test)))

    plt.figure(figsize=(6, 4))
    plt.plot(n_estimators_range, train_scores, label="Train R²")
    plt.plot(n_estimators_range, test_scores, label="Test R²")
    plt.xlabel("Number of Trees")
    plt.ylabel("R² Score")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)

    plt.show()


def plot_actual_vs_predicted(y_test, y_pred, save_path=None):

    plt.figure(figsize=(6, 4))
    plt.scatter(y_test, y_pred, alpha=0.7, edgecolors="k")

    min_val = min(y_test)
    max_val = max(y_test)
    plt.plot([min_val, max_val], [min_val, max_val], 'r--')

    plt.xlabel("Actual Values")
    plt.ylabel("Predicted Values")
    plt.grid(True)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)

    plt.show()


def plot_feature_importance(model, X_test, y_test, feature_names, save_path=None):

    result = permutation_importance(
        model, X_test, y_test,
        n_repeats=20,
        scoring="neg_mean_squared_error",
        random_state=42
    )

    importances = np.abs(result.importances_mean)
    indices = np.argsort(importances)

    plt.figure(figsize=(6, 4))
    plt.barh(range(len(importances)), importances[indices])
    plt.yticks(range(len(importances)), np.array(feature_names)[indices])
    plt.xlabel("Permutation Importance")
    plt.grid(True)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)

    plt.show()


# ==========================
# 5. Main Execution
# ==========================

def main():

    data_path = os.path.join("data", "Gypsum_updated.csv")

    X, y = load_data(data_path)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = train_random_forest(X_train, y_train)

    results, y_pred = evaluate_model(
        model, X_train, X_test, y_train, y_test
    )

    print("\nModel Performance:")
    for key, value in results.items():
        print(f"{key}: {value:.4f}")

    feature_names = [
        "Gypsum Strength",
        "Gypsum Quantity",
        "Water Quantity",
        "Water/Gypsum Ratio",
        "Wheat Straw",
        "CaCl2",
        "Ca(OH)2"
    ]

    os.makedirs("figures", exist_ok=True)

    plot_convergence(
        X_train, y_train, X_test, y_test,
        save_path="figures/convergence_plot.png"
    )

    plot_actual_vs_predicted(
        y_test, y_pred,
        save_path="figures/actual_vs_predicted.png"
    )

    plot_feature_importance(
        model, X_test, y_test,
        feature_names,
        save_path="figures/feature_importance.png"
    )


if __name__ == "__main__":
    main()